# Learn PyDantic

There are three main value props of PyDantic:

  * Validation: If the object ends up getting instantiated, it will have valid data. They make a point about not validating input data, but ensuring that the resulting object will always be valid. I think it refers to the data conversion that they do along the way.
  * Serialization/deserialization
  * JSON schema generation

In [2]:
from pydantic import *
from pprint import pprint
from datetime import datetime

# Model Definition & Construction

Very similar to dataclasses, except all data model classes have to be derived from `BaseModel`.

In [3]:
class Cookie(BaseModel):
    flavor: str
    calories: int

In [4]:
# Can instantiate the object like any other Python object
Cookie(flavor="Chocolate Chip", calories=200)

Cookie(flavor='Chocolate Chip', calories=200)

## Model Config
Model config is the main way to configure type-level behavior of the model. These are controlled by the [ConfigDict](https://docs.pydantic.dev/latest/api/config/) which has a whole bunch of ways to customize the model behavior to be different from the default. There are multiple ways to configure the model -
  * Set the `model_config` attribute in the model.
  * Pass a `config=ConfigDict(...)` to different static validator methods.

In [5]:
class Book(BaseModel):
    title: str
    author: str

    # Model-wide constraint
    # All string fields can have a max length of 5
    model_config = ConfigDict(str_max_length=5)

In [6]:
try:
    # Will error out because title is longer than 5 char
    Book(title="abcdef", author="abc")
except ValidationError as ve:
    pprint(ve.errors())

[{'ctx': {'max_length': 5},
  'input': 'abcdef',
  'loc': ('title',),
  'msg': 'String should have at most 5 characters',
  'type': 'string_too_long',
  'url': 'https://errors.pydantic.dev/2.11/v/string_too_long'}]


In [7]:
try:
    # Will error out because author is longer than 5 chars
    Book(title="abc", author="abcdef")
except ValidationError as ve:
    pprint(ve.errors())

[{'ctx': {'max_length': 5},
  'input': 'abcdef',
  'loc': ('author',),
  'msg': 'String should have at most 5 characters',
  'type': 'string_too_long',
  'url': 'https://errors.pydantic.dev/2.11/v/string_too_long'}]


### Nested Models
Model configs do not apply recursively to nested models. Nested models are models whose fields are other `BaseModel` types.

In [8]:
class A(BaseModel):
    name: str


class B(BaseModel):
    a: A
    title: str

    # This will only apply to str fields of current model, not "recursively" to A
    model_config = ConfigDict(str_max_length=4)

In [9]:
B(title="aaaa", a=A(name="A very long long name"))

B(a=A(name='A very long long name'), title='aaaa')

### Model Heirarchy
If I am building a heirarchy of models, then any model config I set on the top level model is inherited by all the derived models.

In [10]:
class Media(BaseModel):
    model_config = ConfigDict(str_to_upper=True)


class Post(Media):
    title: str


class ShortForm(Post):
    subtitle: str

In [11]:
Post(title="Should I just watch the sunset?")

Post(title='SHOULD I JUST WATCH THE SUNSET?')

In [12]:
ShortForm(title="Should I just watch the sunset?", subtitle="Or embrace the grind?")

ShortForm(title='SHOULD I JUST WATCH THE SUNSET?', subtitle='OR EMBRACE THE GRIND?')

## Fields
The [`Field`](https://docs.pydantic.dev/latest/api/fields/) function is used to set constraints on a per field basis. The linked documentation has all the different constraints/rules I can add to a field.

In [13]:
# Here the constraint only applies to a specific field
class User(BaseModel):
    id: str
    password: str = Field(min_length=3, max_length=8)

In [14]:
# user id can be smaller than 3...
User(id="a", password="abc")

User(id='a', password='abc')

In [15]:
# ...or longer than 8
User(id="abcdefghi", password="abc")

User(id='abcdefghi', password='abc')

In [16]:
# but password has to be between 3 and 8
try:
    User(id="a", password="a")
except ValidationError as ve:
    pprint(ve.errors())

[{'ctx': {'min_length': 3},
  'input': 'a',
  'loc': ('password',),
  'msg': 'String should have at least 3 characters',
  'type': 'string_too_short',
  'url': 'https://errors.pydantic.dev/2.11/v/string_too_short'}]


In [17]:
try:
    User(id="a", password="abcdefghi")
except ValidationError as ve:
    pprint(ve.errors())

[{'ctx': {'max_length': 8},
  'input': 'abcdefghi',
  'loc': ('password',),
  'msg': 'String should have at most 8 characters',
  'type': 'string_too_long',
  'url': 'https://errors.pydantic.dev/2.11/v/string_too_long'}]


`Field()` returns a `FieldInfo` object that I can reuse to add different constraints to different models. The [PyDantic documentation](https://docs.pydantic.dev/latest/concepts/fields/#the-annotated-pattern) states that I should prefer using [Python Annotations](https://docs.python.org/3/library/typing.html#typing.Annotated) to add meta-data to my fields. One of the uses they cite is to build reusable types. But I think I'll stick to using `Field` function as not to mix Python typing esoterica with Pydantic. I can still make reusable types as demonstrated below.

In [18]:
MediumLen = Field(min_length=3, max_length=8)
MediumLen

FieldInfo(annotation=NoneType, required=True, metadata=[MinLen(min_length=3), MaxLen(max_length=8)])

In [19]:
class Song(BaseModel):
    title: str = MediumLen
    artist: str


class PillBox(BaseModel):
    contents: list[str] = MediumLen
    capacity: int

In [20]:
Song(title="Dracula", artist="Tame Impala")

Song(title='Dracula', artist='Tame Impala')

In [21]:
try:
    Song(title="The Fate of Ophelia", artist="Taylor Swift")
except ValidationError as ve:
    pprint(ve.errors())

[{'ctx': {'max_length': 8},
  'input': 'The Fate of Ophelia',
  'loc': ('title',),
  'msg': 'String should have at most 8 characters',
  'type': 'string_too_long',
  'url': 'https://errors.pydantic.dev/2.11/v/string_too_long'}]


In [22]:
PillBox(contents=["one", "two", "three", "four"], capacity=8)

PillBox(contents=['one', 'two', 'three', 'four'], capacity=8)

In [23]:
try:
    PillBox(contents=["one"], capacity=10)
except ValidationError as ve:
    pprint(ve.errors())

[{'ctx': {'actual_length': 1, 'field_type': 'List', 'min_length': 3},
  'input': ['one'],
  'loc': ('contents',),
  'msg': 'List should have at least 3 items after validation, not 1',
  'type': 'too_short',
  'url': 'https://errors.pydantic.dev/2.11/v/too_short'}]


In [24]:
try:
    PillBox(contents=["one"] * 10, capacity=10)
except ValidationError as ve:
    pprint(ve.errors())

[{'ctx': {'actual_length': 10, 'field_type': 'List', 'max_length': 8},
  'input': ['one',
            'one',
            'one',
            'one',
            'one',
            'one',
            'one',
            'one',
            'one',
            'one'],
  'loc': ('contents',),
  'msg': 'List should have at most 8 items after validation, not 10',
  'type': 'too_long',
  'url': 'https://errors.pydantic.dev/2.11/v/too_long'}]


## Default Values
As far as I can tell, there are three ways to set default values -
  * with the `field_name: type = default_val` syntax
  * setting the `Field.default` attribute with a value
  * setting the `Field.default_factory` attribute with a function. This is an "advanced" functionality that is explored in the notebook discussing validation as it is closely tied to how to customize validation.

In [25]:
class DB(BaseModel):
    connstr: str = ":memory"
    major_version: int = Field(default=2)

In [26]:
DB()

DB(connstr=':memory', major_version=2)

## Basic Validation

Model validation happens at construction time. Models can be constructed in the following four ways -
* Using kwargs based constructors
* Using `model_validate()` when the input is a `dict`
* Using `model_validate_json()` when the input is a JSON string
* Using `model_validate_strings()` when the input is a `dict[str, str]`

In [27]:
# Using a simple keyword ctor
chocolate_chip = Cookie(flavor="Chocolate Chip", calories=200)
chocolate_chip

Cookie(flavor='Chocolate Chip', calories=200)

In [28]:
# Using kwargs ctor
snicker_doodle_data = {"flavor": "Snicker Doodle", "calories": 220}
Cookie(**snicker_doodle_data)

Cookie(flavor='Snicker Doodle', calories=220)

In [29]:
# Loss-less data conversion from "180" to 180.
# The IDE type checker will complain
Cookie(flavor="Oatmeal Raisin", calories="180")  # type: ignore

Cookie(flavor='Oatmeal Raisin', calories=180)

In [30]:
# Providing invalid data to the constructor will error out
try:
    Cookie(flavor="Chocolate Chip", calories="high")  # type: ignore
except ValidationError as ve:
    print(ve)
    for err in ve.errors():
        pprint(err)

1 validation error for Cookie
calories
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='high', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/int_parsing
{'input': 'high',
 'loc': ('calories',),
 'msg': 'Input should be a valid integer, unable to parse string as an integer',
 'type': 'int_parsing',
 'url': 'https://errors.pydantic.dev/2.11/v/int_parsing'}


In [31]:
# When I have a Python dict with the data, use model_validate
oatmeal_raisin_data = {"flavor": "Oatmeal Raisin", "calories": 180}
Cookie.model_validate(oatmeal_raisin_data)

Cookie(flavor='Oatmeal Raisin', calories=180)

In [32]:
# When I have JSON data, use model_validate_json
gingerbread_data = """
{
    "flavor": "Gingerbread",
    "calories": 100
}
"""
Cookie.model_validate_json(gingerbread_data)

Cookie(flavor='Gingerbread', calories=100)

In [33]:
# When I have a Python dicts with string key/values use model_validate_strings
pink_sugar_data = {"flavor": "Pink Sugar", "calories": "300"}
Cookie.model_validate_strings(pink_sugar_data)

Cookie(flavor='Pink Sugar', calories=300)

### Bypass validation
I can also instantiate models without doing any validation using the static method `model_construct`. The documentation mentions three use cases for this -

* Performance reasons: when working with complex data, as long as it is known to be valid.
* When validator functions (covered later) are non-idempotent.
* Impure validator functions that have side effects that I don't want to trigger.

In [34]:
# Using model_construct will bypass all validation
Cookie.model_construct(flavor="Choclate Chip", calories="high")

Cookie(flavor='Choclate Chip', calories='high')

### Loss-less Data Conversion

During object creation, PyDantic will perform any loss-less data conversion if needed. The default behavior is to convert an input of a general type to a model attribute of a more specific type as long as that conversion can happen without loss of data. This [conversion table](https://docs.pydantic.dev/latest/concepts/conversion_table/) lists how various types are converted to what.

The documentation states that I should choose to my field types as specific as possible. This makes sense because PyDantic is implemented in Rust and specific field types make it possible to efficiently allocate memory.

The default data conversion behavior can be customized at three different levels:
  * Per field: Customize the behavior differently for different fields in the same model by setting `Field.strict` attribute.
  * Per model: Customize the behavior of all the fields in the model in one shot by setting `ConfigDict.strict` attribute.
  * Per validation call: Only alter the default behavior for one particular validation call by passing the `strict=True` flag.

In [35]:
class IntValue(BaseModel):
    value: int

In [36]:
# OK - no conversion required
IntValue(value=1)

IntValue(value=1)

In [37]:
# OK - This float value can be converted to int without any loss
IntValue(value=1.0)  # type: ignore

IntValue(value=1)

In [38]:
# Can set the strict flag in the validator method so this particular call
# will error out, all others will still do the data conversion
try:
    IntValue.model_validate({"value": 1.0}, strict=True)  # type: ignore
except ValidationError as ve:
    print(ve)
    pprint(ve.errors())

1 validation error for IntValue
value
  Input should be a valid integer [type=int_type, input_value=1.0, input_type=float]
    For further information visit https://errors.pydantic.dev/2.11/v/int_type
[{'input': 1.0,
  'loc': ('value',),
  'msg': 'Input should be a valid integer',
  'type': 'int_type',
  'url': 'https://errors.pydantic.dev/2.11/v/int_type'}]


In [39]:
# ERROR - This float cannot be converted without any loss
try:
    IntValue(value=1.1)  # type: ignore
except ValidationError as ve:
    print(ve)
    for err in ve.errors():
        pprint(err)

1 validation error for IntValue
value
  Input should be a valid integer, got a number with a fractional part [type=int_from_float, input_value=1.1, input_type=float]
    For further information visit https://errors.pydantic.dev/2.11/v/int_from_float
{'input': 1.1,
 'loc': ('value',),
 'msg': 'Input should be a valid integer, got a number with a fractional part',
 'type': 'int_from_float',
 'url': 'https://errors.pydantic.dev/2.11/v/int_from_float'}


In [40]:
# OK - This string can be parsed into an int without any loss
IntValue(value="1")  # type: ignore

IntValue(value=1)

In [41]:
# ERROR - This string cannot be parsed as an int without any loss
try:
    IntValue(value="1.1")  # type: ignore
except ValidationError as ve:
    print(ve)
    for err in ve.errors():
        pprint(err)

1 validation error for IntValue
value
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='1.1', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/int_parsing
{'input': '1.1',
 'loc': ('value',),
 'msg': 'Input should be a valid integer, unable to parse string as an integer',
 'type': 'int_parsing',
 'url': 'https://errors.pydantic.dev/2.11/v/int_parsing'}


In [42]:
# Setting individual field attributes to strict will prevent data conversion for that
# particular field, all other fields will be converted as usual
class Fraction(BaseModel):
    numerator: int
    denominator: int = Field(strict=True)

In [43]:
# OK - numerator will be converted
Fraction(numerator=1.0, denominator=1)  # type: ignore

Fraction(numerator=1, denominator=1)

In [44]:
# ERROR - denominator will not be converted
try:
    Fraction(numerator=1, denominator=1.0)  # type: ignore
except ValidationError as ve:
    print(ve)
    pprint(ve.errors())

1 validation error for Fraction
denominator
  Input should be a valid integer [type=int_type, input_value=1.0, input_type=float]
    For further information visit https://errors.pydantic.dev/2.11/v/int_type
[{'input': 1.0,
  'loc': ('denominator',),
  'msg': 'Input should be a valid integer',
  'type': 'int_type',
  'url': 'https://errors.pydantic.dev/2.11/v/int_type'}]


In [45]:
# Setting the strict flag in the model config will prevent all fields from being
# converted,
class RGB(BaseModel):
    red: int
    green: int
    blue: int

    model_config = ConfigDict(strict=True)

In [46]:
RGB(red=125, green=125, blue=125)

RGB(red=125, green=125, blue=125)

In [47]:
try:
    RGB(red=125.0, green=125, blue=125)  # type: ignore
except ValidationError as ve:
    print(ve)
    pprint(ve.errors())

1 validation error for RGB
red
  Input should be a valid integer [type=int_type, input_value=125.0, input_type=float]
    For further information visit https://errors.pydantic.dev/2.11/v/int_type
[{'input': 125.0,
  'loc': ('red',),
  'msg': 'Input should be a valid integer',
  'type': 'int_type',
  'url': 'https://errors.pydantic.dev/2.11/v/int_type'}]


## Object Behavior

The default behavior of data models is:
* Mutable. This can be changed by `ConfigDict.frozen` attribute.
* No validation after instantiation. TODO: How is this changed?
* Any extra attributes provided to the constructor are silently swallowed. This can be changed by `ConfigDict.extra` attribute.

In [48]:
# Object is mutable
chocolate_chip.flavor = "Double Chocolate Chip"
chocolate_chip

Cookie(flavor='Double Chocolate Chip', calories=200)

In [49]:
# Once the object exists, no validation is done in setters
# The type checker will complain
chocolate_chip.calories = "high"  # type: ignore
chocolate_chip

Cookie(flavor='Double Chocolate Chip', calories='high')

In [50]:
# Any extra attributes provided during construction are silently ignored
# Again the IDE type checker will complain, but the code will run
Cookie(flavor="Chocolate Chip", calories=200, serving_size=2, sugar=30)

Cookie(flavor='Chocolate Chip', calories=200)

### Frozen Models
There are two ways of "freezing" a model, or making a model pseudo-immutable.
  * Using the model config
  * Using an attribute on the `BaseModel`

I prefer using the model config because it keeps my programming model consistent.

In [51]:
class ImmutableCookie(BaseModel):
    flavor: str
    calories: int

    model_config = ConfigDict(frozen=True)

In [52]:
ic = ImmutableCookie(flavor="Chocolate Chip", calories=200)

In [53]:
try:
    ic.flavor = "Double Chocolate Chip"
except ValidationError as ve:
    print(ve)
    pprint(ve.errors())

1 validation error for ImmutableCookie
flavor
  Instance is frozen [type=frozen_instance, input_value='Double Chocolate Chip', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/frozen_instance
[{'input': 'Double Chocolate Chip',
  'loc': ('flavor',),
  'msg': 'Instance is frozen',
  'type': 'frozen_instance',
  'url': 'https://errors.pydantic.dev/2.11/v/frozen_instance'}]


### Extra Fields
By default any extra fields specified during objec construction time are swallowed silently, no errors, but no sign of the data either. I can change this with the `extra` model config which can have three values -
  * `ignore`: silently ignore, which is the default behavior
  * `allow`: retain all extra data
  * `forbid`: throws an error upon seeing any extra data

In [54]:
class User(BaseModel):
    id: int
    name: str

In [55]:
# By default any extra fields are silently ignored
User(id=1, name="AP", hobbies=["Car Racing", "Video Games"])  # type: ignore

User(id=1, name='AP')

In [56]:
# Lets retain all the extra data
class ChillUser(BaseModel):
    id: int
    name: str

    model_config = ConfigDict(extra="allow")

In [57]:
ChillUser(id=1, name="APTG", hobbies=["Car Racing, Video Games"])  # type: ignore

ChillUser(id=1, name='APTG', hobbies=['Car Racing, Video Games'])

In [58]:
# Lets forbid any extra data
class NoChillUser(BaseModel):
    id: int
    name: str

    model_config = ConfigDict(extra="forbid")

In [59]:
try:
    NoChillUser(id=1, name="AP", hobbies=["Car Racing", "Video Game"])  # type: ignore
except ValidationError as ve:
    print(ve)
    pprint(ve.errors())

1 validation error for NoChillUser
hobbies
  Extra inputs are not permitted [type=extra_forbidden, input_value=['Car Racing', 'Video Game'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/extra_forbidden
[{'input': ['Car Racing', 'Video Game'],
  'loc': ('hobbies',),
  'msg': 'Extra inputs are not permitted',
  'type': 'extra_forbidden',
  'url': 'https://errors.pydantic.dev/2.11/v/extra_forbidden'}]


## Union field types

If a field is a union of types, then the validation mechanism will choose the best type for the input, called "smart mode". There are ways to customize this but for the most part the default behavior will suffice. In the example below, even though `"12"` would've been a valid integer with loss-less data conversion, it is still taken as a string because string is a better type match.

In [60]:
class User(BaseModel):
    id: int | str

In [61]:
User(id="12")

User(id='12')

## Date field types

Support for `datetime` object types. Even though Python's JSON module does not handle datetimes well, PyDantic's JSON support does. Loss-less conversion of -
* String datetimes as long as they are in ISO format
* Timestamps

In [62]:
class Holiday(BaseModel):
    name: str
    date: datetime

In [63]:
Holiday(name="Thanksgiving", date=datetime(2025, 11, 27))

Holiday(name='Thanksgiving', date=datetime.datetime(2025, 11, 27, 0, 0))

In [64]:
# Automatically parses an ISO format date string into a datetime object
Holiday(name="Thanksgiving", date="2025-11-27T07:54")  # type: ignore

Holiday(name='Thanksgiving', date=datetime.datetime(2025, 11, 27, 7, 54))

In [65]:
# Does not work with any other types of string date formats
try:
    Holiday(name="Thanksgiving", date="Thursday, November 27, 2025")  # type: ignore
except ValidationError as ve:
    pprint(ve)
    for err in ve.errors():
        pprint(err)

1 validation error for Holiday
date
  Input should be a valid datetime or date, invalid character in year [type=datetime_from_date_parsing, input_value='Thursday, November 27, 2025', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/datetime_from_date_parsing
{'ctx': {'error': 'invalid character in year'},
 'input': 'Thursday, November 27, 2025',
 'loc': ('date',),
 'msg': 'Input should be a valid datetime or date, invalid character in year',
 'type': 'datetime_from_date_parsing',
 'url': 'https://errors.pydantic.dev/2.11/v/datetime_from_date_parsing'}


In [66]:
# Even supports dates in JSON
thanksgiving_data = """
{
    "name": "Thanksgiving",
    "date": "2025-11-27"
}
"""
Holiday.model_validate_json(thanksgiving_data)

Holiday(name='Thanksgiving', date=datetime.datetime(2025, 11, 27, 0, 0))

In [67]:
# Will convert timestamps to datetime objects as well
ts = datetime(2025, 11, 27).timestamp()
Holiday(name="Thanksgiving", date=ts)  # type: ignore

Holiday(name='Thanksgiving', date=datetime.datetime(2025, 11, 27, 8, 0, tzinfo=TzInfo(UTC)))

## Custom Types

Pydantic has a long list of [custom types](https://docs.pydantic.dev/latest/api/types/#pydantic.types) that are pretty versatile. I can use them to control the ser/der of a specific field, control how the field will be validated, add additional methods on the type, etc.

In [68]:
class MyModel(BaseModel):
    size: ByteSize

In [69]:
MyModel(size=52000).size  # type: ignore

52000

In [70]:
MyModel(size="3000 KiB").size  # type: ignore

3072000

In [71]:
m = MyModel(size="50 PB")  # type: ignore

In [72]:
m.size.human_readable()

'44.4PiB'

In [73]:
m.size.human_readable(decimal=True)

'50.0PB'

In [74]:
m.size.human_readable(separator=" ")

'44.4 PiB'

In [75]:
m.size.to("TiB")

45474.73508864641

## Arbitrary Field Types
Fields in Pydantic models can only have "Pydantic Compatible" types. AFAIK - this means they can be -
  * Primitive types
  * Types defined in Python's stdlib
  * Types wrapped in the special `InstanceOf` type
  * Children of `BaseModel`

I can add any aribtrary types with the `ConfigDict.aribtrary_types_allowed` setting, but that comes with a cost - either validaiton suffers or I have to write extra code code to make everything work. It will not work OOB.

In [76]:
class IsABaseModel(BaseModel):
    cookie: int


class NotABaseModel:
    def __init__(self, cookie: int):
        self.cookie = cookie

In [77]:
# I can have a field of type `IsABaseModel`
class Model1(BaseModel):
    value: IsABaseModel

In [78]:
Model1(value=IsABaseModel(cookie=10))

Model1(value=IsABaseModel(cookie=10))

In [79]:
# I cannot have a field of type `NotABaseModel`...
try:

    class Model2(BaseModel):
        value: NotABaseModel

except PydanticSchemaGenerationError as err:
    print(err.message)

Unable to generate pydantic-core schema for <class '__main__.NotABaseModel'>. Set `arbitrary_types_allowed=True` in the model_config to ignore this error or implement `__get_pydantic_core_schema__` on your type to fully support it.

If you got this error by calling handler(<some type>) within `__get_pydantic_core_schema__` then you likely need to call `handler.generate_schema(<some type>)` since we do not call `__get_pydantic_core_schema__` on `<some type>` otherwise to avoid infinite recursion.


In [80]:
# ...unless I wrap it inside InstanceOf
class Model2(BaseModel):
    value: InstanceOf[NotABaseModel]

In [81]:
Model2(value=NotABaseModel(1))

Model2(value=<__main__.NotABaseModel object at 0x7f0fd427c6e0>)

In [82]:
# ...or I configure the model to allow arbitrary types
class Model3(BaseModel):
    value: NotABaseModel

    model_config = ConfigDict(arbitrary_types_allowed=True)

In [83]:
Model3(value=NotABaseModel(1))

Model3(value=<__main__.NotABaseModel object at 0x7f0fd449d310>)

In [84]:
# Standard Python types are allowed
from pathlib import Path


class FilePath(BaseModel):
    file: Path

In [85]:
FilePath(file=Path.home())

FilePath(file=PosixPath('/home/avilay'))

In [86]:
# Loss-less data conversion works as well
FilePath(file="/home/avilay/Documents")  # type: ignore

FilePath(file=PosixPath('/home/avilay/Documents'))

## Type Adapters and Lists

According to the [documentation](https://docs.pydantic.dev/latest/concepts/type_adapter/) `TypeAdapters` work with non-base model types as well as lists of base model types, but I did not find any examples of the former. For now I'll just jot down the usage for lists. The basic idea is that, if I have a list of base models, I can still validate and instantiate the right types without having to create another BaseModel just for the list.

In [87]:
Cookies = TypeAdapter(list[Cookie])
cookies_data = [
    {"flavor": "Chocolate Chip", "calories": 200},
    {"flavor": "Snicker Doodle", "calories": 220},
]
cookies = Cookies.validate_python(cookies_data)
cookies

[Cookie(flavor='Chocolate Chip', calories=200),
 Cookie(flavor='Snicker Doodle', calories=220)]

In [89]:
try:
    Cookies.validate_python(
        [
            {"flavor": "Chocolate Chip", "calories": 200.2},
            {"flavor": "Oatmeal Raisin", "calories": 220},
        ]
    )
except ValidationError as ve:
    pprint(ve.errors())

[{'input': 200.2,
  'loc': (0, 'calories'),
  'msg': 'Input should be a valid integer, got a number with a fractional part',
  'type': 'int_from_float',
  'url': 'https://errors.pydantic.dev/2.11/v/int_from_float'}]


## Forward References

The following two forward references scenarios are supported:
  * Define a model which references another model that has not been defined yet.
  * Recursive references, where a model references itself, but I still have to import `annotations` from `__future__`.

While cyclical references, where an object references another object are allowed in Python, they are not allowed in Pydantic.

In [95]:
from __future__ import annotations


# Reference a model before defining it
# This seems to work sometimes in the notebook, sometimes not.
class ModelX(BaseModel):
    y: ModelY  # type: ignore


class ModelY(BaseModel):
    name: str

In [76]:
from __future__ import annotations


# Recursively reference the model
class Node(BaseModel):
    data: int
    next: Node | None = None  # type: ignore

In [77]:
Node(data=10, next=Node(data=20))

Node(data=10, next=Node(data=20, next=None))

In [81]:
from typing import Any

# Define a cyclic model where x points to y and y in turns points back to x
x: dict[str, Any] = {"data": 10}
y: dict[str, Any] = {"data": 20}
x["next"] = y
y["next"] = x

In [82]:
# This type of cyclic dependency is valid in Python
x

{'data': 10, 'next': {'data': 20, 'next': {...}}}

In [83]:
# But Pydantic does not like it
try:
    Node.model_validate(x)
except ValidationError as ve:
    print(ve)
    pprint(ve.errors())

1 validation error for Node
next.next
  Recursion error - cyclic reference detected [type=recursion_loop, input_value={'data': 10, 'next': {'data': 20, 'next': {...}}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/recursion_loop
[{'input': {'data': 10,
            'next': {'data': 20,
                     'next': <Recursion on dict with id=140017509345536>}},
  'loc': ('next', 'next'),
  'msg': 'Recursion error - cyclic reference detected',
  'type': 'recursion_loop',
  'url': 'https://errors.pydantic.dev/2.11/v/recursion_loop'}]


## Model Copying

When initializing a model object, all the input is copied, even reference types like `list` are copied. However, when I copy a model object using `model_copy()`, the underlying reference types are **not** copied, they are merely rerferenced. So if I change the list in the source object, the change will be visible in the destination object as well. I can change this behavior by setting the `deep=True`. This will make a copy of refernce types.

In [90]:
class PillBox(BaseModel):
    contents: list[str]
    capacity: int

In [91]:
box = ["one", "two"]

In [94]:
pb1 = PillBox(contents=box, capacity=10)
pb1

PillBox(contents=['one', 'two'], capacity=10)

In [95]:
pb2 = PillBox(contents=box, capacity=5)
pb2

PillBox(contents=['one', 'two'], capacity=5)

In [99]:
box.append("three")
box

['one', 'two', 'three', 'three']

In [100]:
pb1

PillBox(contents=['one', 'two'], capacity=10)

In [101]:
pb2

PillBox(contents=['one', 'two'], capacity=5)

In [102]:
pb3 = pb1.model_copy()
pb3

PillBox(contents=['one', 'two'], capacity=10)

In [103]:
pb3.contents.append("three")
pb3

PillBox(contents=['one', 'two', 'three'], capacity=10)

In [104]:
pb1

PillBox(contents=['one', 'two', 'three'], capacity=10)

In [105]:
pb4 = pb2.model_copy(deep=True)
pb4

PillBox(contents=['one', 'two'], capacity=5)

In [106]:
pb4.contents.append("three")
pb4

PillBox(contents=['one', 'two', 'three'], capacity=5)

In [107]:
pb2

PillBox(contents=['one', 'two'], capacity=5)

## Generic Models

Works pretty much like generic types in Python itself. Supports the new syntax and all. When using bounded types, make sure to specify the type parameter when instantiating the data object. If done incorrectly, it might lead to data loss as shown below.

In [90]:
class Snack[T: Cookie](BaseModel):
    cookie: T
    serving_size: int


class LowFatCookie(Cookie):
    carbs_gms: int

In [91]:
Snack[LowFatCookie](
    cookie=LowFatCookie(flavor="Chocolate Chip", calories=100, carbs_gms=25),
    serving_size=2,
)

Snack[LowFatCookie](cookie=LowFatCookie(flavor='Chocolate Chip', calories=100, carbs_gms=25), serving_size=2)

In [92]:
# OK: Even when the type parameter is left out, it will still work because there are on wonky default types.
Snack(
    cookie=LowFatCookie(flavor="Chocolate Chip", calories=100, carbs_gms=25),
    serving_size=2,
)

Snack(cookie=LowFatCookie(flavor='Chocolate Chip', calories=100, carbs_gms=25), serving_size=2)

In [93]:
# Type param is Cookie even though the data has LowFatCookie. carbs_gms is lost.
Snack[Cookie](
    cookie=LowFatCookie(flavor="Chocolate Chip", calories=100, carbs_gms=25),
    serving_size=2,
)

Snack[Cookie](cookie=LowFatCookie(flavor='Chocolate Chip', calories=100, carbs_gms=25), serving_size=2)

In [94]:
Snack[Cookie](
    cookie=LowFatCookie(flavor="Chocolate Chip", calories=100, carbs_gms=25),
    serving_size=2,
).model_dump()

{'cookie': {'flavor': 'Chocolate Chip', 'calories': 100}, 'serving_size': 2}